In [1]:
#****************************************************************************
# (C) Cloudera, Inc. 2020-2023
#  All rights reserved.
#
#  Applicable Open Source License: GNU Affero General Public License v3.0
#
#  NOTE: Cloudera open source products are modular software products
#  made up of hundreds of individual components, each of which was
#  individually copyrighted.  Each Cloudera open source product is a
#  collective work under U.S. Copyright Law. Your license to use the
#  collective work is as provided in your written agreement with
#  Cloudera.  Used apart from the collective work, this file is
#  licensed for your use pursuant to the open source license
#  identified above.
#
#  This code is provided to you pursuant a written agreement with
#  (i) Cloudera, Inc. or (ii) a third-party authorized to distribute
#  this code. If you do not have a written agreement with Cloudera nor
#  with an authorized and properly licensed third party, you do not
#  have any rights to access nor to use this code.
#
#  Absent a written agreement with Cloudera, Inc. (“Cloudera”) to the
#  contrary, A) CLOUDERA PROVIDES THIS CODE TO YOU WITHOUT WARRANTIES OF ANY
#  KIND; (B) CLOUDERA DISCLAIMS ANY AND ALL EXPRESS AND IMPLIED
#  WARRANTIES WITH RESPECT TO THIS CODE, INCLUDING BUT NOT LIMITED TO
#  IMPLIED WARRANTIES OF TITLE, NON-INFRINGEMENT, MERCHANTABILITY AND
#  FITNESS FOR A PARTICULAR PURPOSE; (C) CLOUDERA IS NOT LIABLE TO YOU,
#  AND WILL NOT DEFEND, INDEMNIFY, NOR HOLD YOU HARMLESS FOR ANY CLAIMS
#  ARISING FROM OR RELATED TO THE CODE; AND (D)WITH RESPECT TO YOUR EXERCISE
#  OF ANY RIGHTS GRANTED TO YOU FOR THE CODE, CLOUDERA IS NOT LIABLE FOR ANY
#  DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, PUNITIVE OR
#  CONSEQUENTIAL DAMAGES INCLUDING, BUT NOT LIMITED TO, DAMAGES
#  RELATED TO LOST REVENUE, LOST PROFITS, LOSS OF INCOME, LOSS OF
#  BUSINESS ADVANTAGE OR UNAVAILABILITY, OR LOSS OR CORRUPTION OF
#  DATA.
#
# #  Author(s): Paul de Fusco
#***************************************************************************/

## PLEASE USER JUPYTERLAB - Python 3.12 and SPARK 3.2+ RUNTIME for this demo

In [1]:
#pip install the stuff you need
# pip install \
# mlflow==2.19.0 \
# torch==2.5.1 \
# torchvision==0.20.1 \
# tensorflow==2.18.0 \
# xgboost==3.1.2 \
# scikit-learn==1.8.0 \
# transformers==4.46.3 \
# numpy==1.26.4 \
# pillow==10.4.0 \
# pandas==2.3.3
!pip install -q \
mlflow==2.19.0 \
xgboost==3.1.2 \
scikit-learn==1.8.0 \
numpy==1.26.4 \
pillow==10.4.0 \
pandas==2.3.3 \
onnxmltools \
onnxruntime

In [2]:
import os, warnings, sys, logging
import mlflow
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score
import mlflow.sklearn
from xgboost import XGBClassifier
from datetime import date
import cml.data_v1 as cmldata
import pyspark.pandas as ps

import onnxmltools
from urllib.parse import urlparse
from sklearn.preprocessing import FunctionTransformer
from mlflow.models import infer_signature
#from onnxconverter_common import FloatTensorType
from onnxmltools.convert.common.data_types import FloatTensorType, Int64TensorType
from onnxmltools import convert_xgboost 

In [3]:
USERNAME = os.environ["PROJECT_OWNER"]
DBNAME = os.environ["DBNAME_PREFIX"]+"_"+USERNAME
CONNECTION_NAME = os.environ["SPARK_CONNECTION_NAME"]

DATE = date.today()
EXPERIMENT_NAME = "xgb-cc-fraud-{0}".format(USERNAME)

mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='/home/cdsw/.experiments/4r3f-ugr9-jvia-0eqa', creation_time=None, experiment_id='4r3f-ugr9-jvia-0eqa', last_update_time=None, lifecycle_stage='active', name='xgb-cc-fraud-user020', tags={}>

In [4]:
conn = cmldata.get_connection(CONNECTION_NAME)
spark = conn.get_spark_session()

Setting spark.hadoop.yarn.resourcemanager.principal to user020


Spark Application Id:spark-application-1775029811386


In [5]:
#df.columns

In [6]:
df_from_sql = ps.read_table('{0}.transactions_{1}'.format(DBNAME, USERNAME))
df = df_from_sql.to_pandas()

y = df["fraud_trx"]
X = df.drop(columns=["fraud_trx"])
X.columns = ['f' + str(i) for i in range(len(X.columns))]

test_size = 0.3
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size)

Hive Session ID = 855580fb-c5e6-4ea0-ab5e-51168e8344c1
                                                                                

In [7]:
X_train.dtypes

f0     float32
f1     float32
f2     float32
f3     float32
f4     float32
f5     float32
f6     float32
f7     float32
f8     float32
f9     float32
f10    float32
f11    float32
f12    float32
f13    float32
f14    float32
dtype: object

In [ ]:
USERNAME = os.environ["PROJECT_OWNER"]
model_name = "fraud-clf-onnx-xgboost-{0}".format(USERNAME)
with mlflow.start_run():

    model = XGBClassifier(use_label_encoder=False, eval_metric="logloss")

    # Step 1: cambiar test_size linea 69 y recorrer
    # Step 2: cambiar linea 74, agregar linea 97, y recorrer
      # linea 75: model = XGBClassifier(use_label_encoder=False, max_depth=4, eval_metric="logloss")
      # linea 97: mlflow.log_param("max_depth", 4)
    # Step 3: cambiar linea 74 y 97, agregar linea 98, y recorrer
      # linea 75: model = XGBClassifier(use_label_encoder=False, max_depth=2, max_leaf_nodes=5, eval_metric="logloss")
      # linea 97: mlflow.log_param("max_depth", 2)
      # linea 98: mlflow.log_param("max_leaf_nodes", 5)

    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    print("Accuracy: %.2f%%" % (accuracy * 100.0))
    print("Test Size: %.2f%%" % (test_size * 100.0))

    mlflow.log_param("accuracy", accuracy)
    mlflow.log_param("test_size", test_size)

    # Step 2:
    # Step 3:

    num_features = X_train.shape[1]
    #initial_type = [("input", FloatTensorType([None, num_features]))]

    #schema = [("X", FloatTensorType([None, X_train.shape[1]]))]
    schema = [("input", FloatTensorType([None, num_features]))]
    onnx_model = convert_xgboost(
        model, initial_types = schema)
    
    model_signature = infer_signature(X_train, y_pred)
    #onnx_model = onnxmltools.convert_xgboost(model, initial_types=initial_type)
    #onnxmltools.utils.save_model(onnx_model, "fraud_classifier.onnx")
    mlflow.onnx.log_model(onnx_model, "fraud-clf-onnx-xgboost",
                          registered_model_name=model_name,
                          signature=model_signature)

/home/cdsw/.local/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [07:59:49] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Accuracy: 91.98%
Test Size: 30.00%


Successfully registered model 'fraud-clf-onnx-xgboost-user020'.
